# Connecting Python to MySQL with `mysql-connector-python`

This notebook walks through how a Python program talks to a MySQL database:

1. Open a **connection** to the MySQL server
2. Build the **COMPANY** database from `company_db_initialization_WC.sql`
3. Use a **cursor** to send SQL and read results (`fetchone`, `fetchall`, `fetchmany`)
4. Write **parameterized queries** and see why they prevent SQL injection
5. Change data with `INSERT` / `UPDATE` / `DELETE`, and control it with `commit()` / `rollback()`
6. Handle errors and close everything cleanly

### The big picture

```
Python program ──► connection ──► cursor.execute("SQL") ──► MySQL server
                                         │
             rows (list of tuples) ◄── cursor.fetchall()
```

- A **connection** is the open line to the server (one login session).
- A **cursor** is the object you send SQL through. It also holds the result rows until you fetch them.

## 0. Setup in VS Code

Before running anything:

1. **MySQL Server** must be installed and running. (MySQL Workbench is only a client, so the server has to be running too.)
2. In VS Code, install the **Python** and **Jupyter** extensions (both from Microsoft).
3. Open this `.ipynb` file. Click **Select Kernel** (top-right) → **Python Environments** → pick your Python interpreter.
4. Put `company_db_initialization_WC.sql` in the **same folder** as this notebook.
5. Run a cell with **Shift + Enter**. Run them **in order**, from top to bottom.

> **Tip:** If something goes wrong partway through, use **Restart** in the notebook toolbar, then **Run All**.

## 1. Install and import the connector

`mysql-connector-python` is Oracle's official MySQL driver for Python. The `%pip` magic installs it into **the same environment the notebook kernel is using**. That matters because `pip install` in a terminal can install into a different Python.

We also use **pandas** to display results as tables. It is optional.

In [25]:
%pip install mysql-connector-python pandas

Note: you may need to restart the kernel to use updated packages.


In [27]:
import mysql.connector
from mysql.connector import Error, errorcode
import pandas as pd

print("mysql-connector-python version:", mysql.connector.__version__)

mysql-connector-python version: 26.10.0


## 2. Connect to the MySQL server

`mysql.connector.connect(...)` returns a **connection object**. The main arguments are:

| Argument   | Meaning                                    | Typical value  |
|------------|--------------------------------------------|----------------|
| `host`     | Where the server runs                       | `"localhost"`  |
| `port`     | Server port                                 | `3306`         |
| `user`     | MySQL username                              | `"root"`       |
| `password` | That user's password                         |                |
| `database` | *(optional)* Database to `USE` on connect   | `"COMPANY"`    |

We **don't** pass `database` yet, because COMPANY doesn't exist until we run the script.

> **Don't hard-code passwords** in files you share or commit to Git. `getpass` prompts for the password when the cell runs. In VS Code, the prompt opens at the **top of the window**.

In [36]:
from getpass import getpass

DB_CONFIG = {
    "host": "localhost",
    "port": 3306,
    "user": "root",
    "password": getpass("MySQL password: ")
}

In [37]:
try:
    conn = mysql.connector.connect(**DB_CONFIG)
    print("Connected:", conn.is_connected())
    print("Server version:", conn.server_info)
except Error as e:
    print("Connection failed:", e)

Connected: True
Server version: 8.0.46


**Common connection errors**

| Error                                      | Usual cause                                   |
|--------------------------------------------|-----------------------------------------------|
| `2003: Can't connect to MySQL server`      | Server not running, or wrong host/port        |
| `1045: Access denied for user`             | Wrong username or password                    |
| `1049: Unknown database`                   | `database=` names a database that doesn't exist |

## 3. Your first cursor

Every SQL statement goes through a **cursor**:

1. `cursor = conn.cursor()` creates it
2. `cursor.execute(sql)` sends one statement to the server
3. `cursor.fetchall()` / `fetchone()` reads the results
4. `cursor.close()` releases it

In [44]:
cursor = conn.cursor()

cursor.execute("SELECT VERSION(), CURRENT_USER(), NOW()")
row = cursor.fetchone()      # one row, returned as a tuple
print(row)
print(type(row))

cursor.execute("SHOW DATABASES")
for (db_name,) in cursor.fetchall():   # each row is a 1-tuple
    print(" -", db_name)

('8.0.46', 'root@localhost', datetime.datetime(2026, 11, 18, 9, 24, 22))
<class 'tuple'>
 - company
 - information_schema
 - mysql
 - performance_schema
 - sakila
 - sys
 - world


## 4. Build the COMPANY database from the SQL script

`cursor.execute()` runs **one statement at a time**. The script contains many statements separated by `;`, so we:

1. read the file,
2. split it on `;`,
3. execute each piece in turn,
4. `commit()` so the inserted rows are saved.

> Splitting on `;` works for this script. It would break on scripts that contain `;` inside strings, or that define stored procedures. For those, run the script in MySQL Workbench or with the `mysql` command-line client instead.

⚠️ The script begins with `DROP DATABASE IF EXISTS COMPANY;`, so running this cell **resets the database** to its original state. That's handy when you want to undo your experiments.

In [46]:
SCRIPT_PATH = "company_db_initialization_WC.sql"

def run_sql_script(connection, path):
    """Execute every ;-separated statement in a .sql file."""
    with open(path, encoding="utf-8") as f:
        script = f.read()

    statements = [s.strip() for s in script.split(";") if s.strip()]
    cur = connection.cursor()
    try:
        for stmt in statements:
            cur.execute(stmt)
            if cur.with_rows:        # some statements return rows; read them so the cursor is ready for the next one
                cur.fetchall()
        connection.commit()
        print(f"Executed {len(statements)} statements from {path}")
    except Error as e:
        connection.rollback()
        print("Failed on statement:\n", stmt[:200], "\nError:", e)
        raise
    finally:
        cur.close()

run_sql_script(conn, SCRIPT_PATH)

Executed 16 statements from company_db_initialization_WC.sql


Because the script ran `USE COMPANY;`, this connection now points at COMPANY. Let's confirm and list the tables:

In [48]:
cursor.execute("SELECT DATABASE()")
print("Current database:", cursor.fetchone()[0])

cursor.execute("SHOW TABLES")
print([t for (t,) in cursor.fetchall()])

Current database: company
['department', 'dependent', 'dept_locations', 'employee', 'project', 'works_on']


### Reconnecting directly to COMPANY

From now on you can connect straight to the database with the `database` argument. If you restart the kernel later, you don't need to run the script again. Just run this cell (after the imports and `DB_CONFIG` cells).

In [50]:
cursor.close()
conn.close()

conn = mysql.connector.connect(**DB_CONFIG, database="COMPANY")
cursor = conn.cursor()
print("Connected to:", conn.database)

Connected to: company


## 5. Reading results: `fetchall`, `fetchone`, `fetchmany`

After a `SELECT`, the rows wait in the cursor until you fetch them.

| Method           | Returns                                   | When to use                       |
|------------------|-------------------------------------------|-----------------------------------|
| `fetchall()`     | list of all remaining rows                | small/medium results              |
| `fetchone()`     | the next row (or `None` when finished)    | exactly one row expected          |
| `fetchmany(n)`   | list of up to `n` rows                    | processing large results in chunks|
| `for row in cursor` | one row at a time                      | streaming through results         |

In [55]:
cursor.execute("SELECT Fname, Lname, Salary FROM EMPLOYEE")
rows = cursor.fetchall()

print(type(rows), "with", len(rows), "rows")
for fname, lname, salary in rows:          # tuple unpacking
    print(f"{fname:<10} {lname:<10} {salary:>8}")

<class 'list'> with 8 rows
John       Smith         30000
Franklin   Wong          40000
Joyce      English       25000
Ramesh     Narayan       38000
James      Borg          55000
Jennifer   Wallace       43000
Ahmad      Jabbar        25000
Alicia     Zelaya        25000


Notice that `Salary` comes back as `Decimal('30000')`. The connector converts each MySQL type to a matching Python type:

| MySQL          | Python               |
|----------------|----------------------|
| `INT`          | `int`                |
| `DECIMAL`      | `decimal.Decimal`    |
| `VARCHAR/CHAR` | `str`                |
| `DATE`         | `datetime.date`      |
| `NULL`         | `None`               |

In [57]:
cursor.execute("SELECT Ssn, Bdate, Salary, Super_ssn FROM EMPLOYEE WHERE Lname = 'Borg'")
row = cursor.fetchone()
for value in row:
    print(repr(value), "->", type(value).__name__)

'888665555' -> str
datetime.date(1937, 11, 10) -> date
Decimal('55000') -> Decimal
None -> NoneType


In [58]:
# fetchone() in a loop: returns None once the rows run out
cursor.execute("SELECT Dname, Dnumber FROM DEPARTMENT ORDER BY Dnumber")
row = cursor.fetchone()
while row is not None:
    print(row)
    row = cursor.fetchone()

('Headquarters', 1)
('Administration', 4)
('Research', 5)


In [59]:
# fetchmany(n): process the rows in batches
cursor.execute("SELECT Essn, Pno, Hours FROM WORKS_ON")
batch_no = 1
while True:
    batch = cursor.fetchmany(5)
    if not batch:
        break
    print(f"Batch {batch_no}: {batch}")
    batch_no += 1

Batch 1: [('123456789', 1, Decimal('32.5')), ('123456789', 2, Decimal('7.5')), ('333445555', 2, Decimal('10.0')), ('333445555', 3, Decimal('10.0')), ('333445555', 10, Decimal('10.0'))]
Batch 2: [('333445555', 20, Decimal('10.0')), ('453453453', 1, Decimal('20.0')), ('453453453', 2, Decimal('20.0')), ('666884444', 3, Decimal('40.0')), ('888665555', 20, Decimal('16.0'))]
Batch 3: [('987654321', 20, Decimal('15.0')), ('987654321', 30, Decimal('20.0')), ('987987987', 10, Decimal('35.0')), ('987987987', 30, Decimal('5.0')), ('999887777', 10, Decimal('10.0'))]
Batch 4: [('999887777', 30, Decimal('30.0'))]


In [60]:
# Iterate over the cursor directly
cursor.execute("SELECT Pname, Plocation FROM PROJECT")
for pname, ploc in cursor:
    print(f"{pname} is located in {ploc}")

ProductX is located in Bellaire
ProductY is located in Sugarland
ProductZ is located in Houston
Computerization is located in Stafford
Reorganization is located in Houston
Newbenefits is located in Stafford


### Metadata about the last query

- `cursor.column_names` gives the column headers
- `cursor.rowcount` gives the number of rows (for a `SELECT`, only accurate after fetching)
- `cursor.statement` gives the exact SQL that was sent

In [61]:
cursor.execute("SELECT Fname, Lname, Dno FROM EMPLOYEE WHERE Sex = 'F'")
rows = cursor.fetchall()
print("Columns:  ", cursor.column_names)
print("Row count:", cursor.rowcount)
print("Statement:", cursor.statement)

Columns:   ('Fname', 'Lname', 'Dno')
Row count: 3
Statement: SELECT Fname, Lname, Dno FROM EMPLOYEE WHERE Sex = 'F'


> **"Unread result found" error:** if you `execute()` a `SELECT` and don't fetch all of its rows, the next `execute()` on that connection raises `InternalError: Unread result found`. Always fetch everything, or create the cursor with `conn.cursor(buffered=True)`. A buffered cursor reads the whole result into memory as soon as the query runs.

## 6. Dictionary cursors

Tuples force you to remember column positions. With `dictionary=True`, each row is a `dict` keyed by column name instead.

In [62]:
dcur = conn.cursor(dictionary=True)
dcur.execute("SELECT Fname, Lname, Salary FROM EMPLOYEE WHERE Dno = 5")
for emp in dcur.fetchall():
    print(emp)

print()
print(emp["Fname"], "earns", emp["Salary"])
dcur.close();

{'Fname': 'John', 'Lname': 'Smith', 'Salary': Decimal('30000')}
{'Fname': 'Franklin', 'Lname': 'Wong', 'Salary': Decimal('40000')}
{'Fname': 'Joyce', 'Lname': 'English', 'Salary': Decimal('25000')}
{'Fname': 'Ramesh', 'Lname': 'Narayan', 'Salary': Decimal('38000')}

Ramesh earns 38000


## 7. A helper: query → pandas DataFrame

Printing tuples gets tedious. This small helper runs a query and returns the result as a table. `cursor.column_names` provides the headers.

In [63]:
def query_df(sql, params=None):
    """Run a SELECT and return the result as a pandas DataFrame."""
    with conn.cursor() as cur:            # the cursor closes automatically
        cur.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=cur.column_names)

query_df("SELECT * FROM EMPLOYEE")

,Fname,Minit,Lname,Ssn,Bdate,Address,Sex,Salary,Super_ssn,Dno
0,John,B,Smith,123456789,1965-01-09,"731 Fondren, Houston TX",M,30000,333445555,5
1,Franklin,T,Wong,333445555,1965-12-08,"638 Voss, Houston TX",M,40000,888665555,5
2,Joyce,A,English,453453453,1972-07-31,"5631 Rice, Houston TX",F,25000,333445555,5
3,Ramesh,K,Narayan,666884444,1962-09-15,"975 Fire Oak, Humble TX",M,38000,333445555,5
4,James,E,Borg,888665555,1937-11-10,"450 Stone, Houston TX",M,55000,NaN,1
5,Jennifer,S,Wallace,987654321,1941-06-20,"291 Berry, Bellaire TX",F,43000,888665555,4
6,Ahmad,V,Jabbar,987987987,1969-03-29,"980 Dallas, Houston TX",M,25000,987654321,4
7,Alicia,J,Zelaya,999887777,1968-01-19,"3321 Castle, Spring TX",F,25000,987654321,4


## 8. Example queries on COMPANY

These are classic queries on the COMPANY schema. Run each one, then try changing it.

**8.1 Name and address of every employee in the 'Research' department** (a join)

In [64]:
query_df("""
    SELECT E.Fname, E.Lname, E.Address
    FROM   EMPLOYEE E
    JOIN   DEPARTMENT D ON E.Dno = D.Dnumber
    WHERE  D.Dname = 'Research'
""")

,Fname,Lname,Address
0,John,Smith,"731 Fondren, Houston TX"
1,Franklin,Wong,"638 Voss, Houston TX"
2,Joyce,English,"5631 Rice, Houston TX"
3,Ramesh,Narayan,"975 Fire Oak, Humble TX"


**8.2 Each employee with their supervisor's name** (self-join; `LEFT JOIN` keeps Borg, who has no supervisor)

In [65]:
query_df("""
    SELECT E.Fname AS Emp_first, E.Lname AS Emp_last,
           S.Fname AS Sup_first, S.Lname AS Sup_last
    FROM   EMPLOYEE E
    LEFT JOIN EMPLOYEE S ON E.Super_ssn = S.Ssn
""")

,Emp_first,Emp_last,Sup_first,Sup_last
0,John,Smith,Franklin,Wong
1,Franklin,Wong,James,Borg
2,Joyce,English,Franklin,Wong
3,Ramesh,Narayan,Franklin,Wong
4,James,Borg,NaN,NaN
5,Jennifer,Wallace,James,Borg
6,Ahmad,Jabbar,Jennifer,Wallace
7,Alicia,Zelaya,Jennifer,Wallace


**8.3 For each project in Stafford: the controlling department and its manager**

In [66]:
query_df("""
    SELECT P.Pnumber, P.Pname, D.Dname, E.Lname AS Manager, E.Bdate
    FROM   PROJECT P
    JOIN   DEPARTMENT D ON P.Dnum = D.Dnumber
    JOIN   EMPLOYEE  E ON D.Mgr_ssn = E.Ssn
    WHERE  P.Plocation = 'Stafford'
""")

,Pnumber,Pname,Dname,Manager,Bdate
0,10,Computerization,Administration,Wallace,1941-06-20
1,30,Newbenefits,Administration,Wallace,1941-06-20


**8.4 Headcount and salary statistics per department** (`GROUP BY` and aggregates)

In [67]:
query_df("""
    SELECT D.Dname,
           COUNT(*)      AS Num_employees,
           SUM(E.Salary) AS Total_salary,
           AVG(E.Salary) AS Avg_salary,
           MAX(E.Salary) AS Max_salary
    FROM   EMPLOYEE E
    JOIN   DEPARTMENT D ON E.Dno = D.Dnumber
    GROUP BY D.Dname
    ORDER BY Avg_salary DESC
""")

,Dname,Num_employees,Total_salary,Avg_salary,Max_salary
0,Headquarters,1,55000,55000.0000,55000
1,Research,4,133000,33250.0000,40000
2,Administration,3,93000,31000.0000,43000


**8.5 Projects with more than two employees, and their total hours** (`HAVING`)

In [68]:
query_df("""
    SELECT P.Pname, COUNT(*) AS Num_workers, SUM(W.Hours) AS Total_hours
    FROM   PROJECT P
    JOIN   WORKS_ON W ON P.Pnumber = W.Pno
    GROUP BY P.Pname
    HAVING COUNT(*) > 2
""")

,Pname,Num_workers,Total_hours
0,Computerization,3,55.0
1,Newbenefits,3,55.0
2,ProductY,3,37.5
3,Reorganization,3,41.0


**8.6 Employees with no dependents** (`NOT EXISTS`)

In [69]:
query_df("""
    SELECT Fname, Lname
    FROM   EMPLOYEE E
    WHERE  NOT EXISTS (SELECT * FROM DEPENDENT D WHERE D.Essn = E.Ssn)
""")

,Fname,Lname
0,Joyce,English
1,Ramesh,Narayan
2,James,Borg
3,Ahmad,Jabbar
4,Alicia,Zelaya


**8.7 Managers who have at least one dependent**

In [70]:
query_df("""
    SELECT DISTINCT E.Fname, E.Lname
    FROM   EMPLOYEE E
    JOIN   DEPARTMENT DP ON DP.Mgr_ssn = E.Ssn
    JOIN   DEPENDENT  DE ON DE.Essn   = E.Ssn
""")

,Fname,Lname
0,Franklin,Wong
1,Jennifer,Wallace


## 9. Parameterized queries (and why you must use them)

Most real queries depend on a value from the user, such as a department name typed into a form. It's tempting to build the SQL with an f-string. **Don't.**

In [ ]:
# ❌ UNSAFE: user input pasted directly into SQL
user_input = "Research"
sql = f"SELECT Fname, Lname FROM EMPLOYEE E JOIN DEPARTMENT D ON E.Dno = D.Dnumber WHERE D.Dname = '{user_input}'"
print(sql)
query_df(sql)

That works for normal input. But a malicious user could type `' OR '1'='1` instead:

In [ ]:
user_input = "' OR '1'='1"
sql = f"SELECT Fname, Lname, Salary FROM EMPLOYEE E JOIN DEPARTMENT D ON E.Dno = D.Dnumber WHERE D.Dname = '{user_input}'"
print(sql)
query_df(sql)          # returns EVERY employee and salary: SQL injection!

### ✅ The fix: placeholders

Put `%s` in the SQL where each value goes, and pass the values **separately** as a tuple. The connector sends them as data, never as SQL code, and handles quoting and escaping for you.

- Write `%s` for **every** type (numbers, strings, dates). Don't put quotes around it.
- A single value still needs a tuple: `(value,)`. Note the comma.
- Placeholders can only stand in for **values**, not table or column names.

In [ ]:
sql = """
    SELECT Fname, Lname, Salary
    FROM   EMPLOYEE E JOIN DEPARTMENT D ON E.Dno = D.Dnumber
    WHERE  D.Dname = %s
"""

cursor.execute(sql, ("Research",))
print("Normal input:   ", cursor.fetchall())

cursor.execute(sql, ("' OR '1'='1",))
print("Injection input:", cursor.fetchall())   # [] : treated as a literal (nonexistent) name
print("\nSQL actually sent:", cursor.statement)

**Several parameters**, and **named parameters** with `%(name)s` and a dict:

In [ ]:
# Positional: values in the same order as the %s markers
query_df(
    "SELECT Fname, Lname, Salary, Dno FROM EMPLOYEE WHERE Salary BETWEEN %s AND %s AND Dno = %s",
    (25000, 40000, 5),
)

In [ ]:
# Named: easier to read when there are many parameters
params = {"min_sal": 30000, "sex": "M"}
query_df(
    "SELECT Fname, Lname, Sex, Salary FROM EMPLOYEE WHERE Salary >= %(min_sal)s AND Sex = %(sex)s",
    params,
)

In [ ]:
# A reusable function: total hours an employee works across all projects
def hours_for(last_name):
    cursor.execute("""
        SELECT E.Fname, E.Lname, SUM(W.Hours)
        FROM   EMPLOYEE E JOIN WORKS_ON W ON E.Ssn = W.Essn
        WHERE  E.Lname = %s
        GROUP BY E.Ssn, E.Fname, E.Lname
    """, (last_name,))
    return cursor.fetchone()

for name in ["Smith", "Wong", "Borg"]:
    print(hours_for(name))

## 10. Modifying data: INSERT, UPDATE, DELETE

By default, mysql-connector **does not auto-commit**. Your `INSERT`, `UPDATE`, and `DELETE` statements are part of an open **transaction**, which only you (this connection) can see, until you call:

- `conn.commit()` to make the changes permanent, or
- `conn.rollback()` to discard them.

If you close the connection without committing, the changes are lost.

### 10.1 INSERT a single row

In [ ]:
insert_sql = """
    INSERT INTO EMPLOYEE (Fname, Minit, Lname, Ssn, Bdate, Address, Sex, Salary, Super_ssn, Dno)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""
new_emp = ("Richard", "K", "Marini", "653298653", "1972-12-30",
           "98 Oak Forest, Katy TX", "M", 37000, "987654321", 4)

cursor.execute(insert_sql, new_emp)
print("Rows inserted:", cursor.rowcount)
conn.commit()

query_df("SELECT * FROM EMPLOYEE WHERE Ssn = %s", ("653298653",))

### 10.2 INSERT many rows with `executemany`

`executemany()` runs the same statement once for each tuple in a list. For `INSERT`, the connector combines them into a single multi-row statement, which is much faster than a Python loop.

In [ ]:
assignments = [
    ("653298653", 10, 20.0),
    ("653298653", 30, 15.0),
]
cursor.executemany("INSERT INTO WORKS_ON (Essn, Pno, Hours) VALUES (%s, %s, %s)", assignments)
print("Rows inserted:", cursor.rowcount)
conn.commit()

query_df("SELECT * FROM WORKS_ON WHERE Essn = %s", ("653298653",))

### 10.3 UPDATE: give everyone in Research a 10% raise

In [ ]:
cursor.execute("""
    UPDATE EMPLOYEE
    SET    Salary = Salary * 1.10
    WHERE  Dno = (SELECT Dnumber FROM DEPARTMENT WHERE Dname = %s)
""", ("Research",))
print("Rows updated:", cursor.rowcount)
conn.commit()

query_df("SELECT Fname, Lname, Salary FROM EMPLOYEE WHERE Dno = 5")

### 10.4 DELETE

Foreign keys still apply. Marini has rows in `WORKS_ON`, so those must be deleted first.

In [ ]:
cursor.execute("DELETE FROM WORKS_ON WHERE Essn = %s", ("653298653",))
print("WORKS_ON rows deleted:", cursor.rowcount)
cursor.execute("DELETE FROM EMPLOYEE WHERE Ssn = %s", ("653298653",))
print("EMPLOYEE rows deleted:", cursor.rowcount)
conn.commit()

## 11. Transactions: `commit()` vs `rollback()`

A transaction groups several statements so that either **all** of them take effect or **none** do. Here we make a mistake, check it, and undo it.

In [ ]:
cursor.execute("UPDATE EMPLOYEE SET Salary = 0")        # oops, there's no WHERE clause!
print("Rows changed:", cursor.rowcount)
print(query_df("SELECT Fname, Salary FROM EMPLOYEE").head(3), "\n")

conn.rollback()                                         # undo everything since the last commit
print("After rollback:")
print(query_df("SELECT Fname, Salary FROM EMPLOYEE").head(3))

### The standard pattern: commit on success, roll back on any error

The example below moves an employee to a new department and updates their supervisor. Both changes should succeed together. The second statement fails, because department 99 doesn't exist and breaks the foreign key. As a result, the first change is undone as well.

In [ ]:
def transfer_employee(ssn, new_dno, new_super_ssn):
    try:
        cursor.execute("UPDATE EMPLOYEE SET Super_ssn = %s WHERE Ssn = %s", (new_super_ssn, ssn))
        cursor.execute("UPDATE EMPLOYEE SET Dno = %s WHERE Ssn = %s", (new_dno, ssn))
        conn.commit()
        print("Transfer committed.")
    except Error as e:
        conn.rollback()
        print(f"Transfer failed and was rolled back.\n  MySQL error {e.errno}: {e.msg}")

transfer_employee("123456789", 99, "987654321")     # department 99 doesn't exist
query_df("SELECT Fname, Lname, Super_ssn, Dno FROM EMPLOYEE WHERE Ssn = '123456789'")   # unchanged

> **Autocommit:** setting `conn.autocommit = True` commits every statement immediately. That's convenient for quick experiments, but then `rollback()` can no longer undo anything. Leave it off when several changes must succeed or fail together.

## 12. Handling errors

The connector raises `mysql.connector.Error` (or a subclass). Its useful attributes are:

- `e.errno`, the MySQL error number
- `e.sqlstate`, the standard SQLSTATE code
- `e.msg`, the message text

`errorcode` gives the numbers readable names, so you can react to specific errors.

In [ ]:
tests = [
    ("Duplicate primary key", "INSERT INTO DEPARTMENT VALUES ('Test', 5, '888665555', NULL)"),
    ("Unknown table",         "SELECT * FROM EMPLOYEES"),
    ("Syntax error",          "SELEC * FROM EMPLOYEE"),
    ("FK violation",          "INSERT INTO WORKS_ON VALUES ('000000000', 1, 5.0)"),
]

for label, sql in tests:
    try:
        cursor.execute(sql)
    except Error as e:
        if e.errno == errorcode.ER_DUP_ENTRY:
            kind = "ER_DUP_ENTRY"
        elif e.errno == errorcode.ER_NO_SUCH_TABLE:
            kind = "ER_NO_SUCH_TABLE"
        elif e.errno == errorcode.ER_PARSE_ERROR:
            kind = "ER_PARSE_ERROR"
        elif e.errno == errorcode.ER_NO_REFERENCED_ROW_2:
            kind = "ER_NO_REFERENCED_ROW_2"
        else:
            kind = "other"
        print(f"{label:<22} errno={e.errno} ({kind})  {e.msg[:60]}")

conn.rollback()

## 13. Closing up, and using `with` blocks

Cursors and connections hold resources on the server, so close them when you're done. Both work as **context managers**: a `with` block closes them automatically, even if an error occurs inside it.

In [ ]:
with mysql.connector.connect(**DB_CONFIG, database="COMPANY") as c:
    with c.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM EMPLOYEE")
        print("Employees:", cur.fetchone()[0])
# both are closed here

print("Still connected?", c.is_connected())

> **Note:** leaving a `with` block closes the connection, but it does **not** commit for you. Call `c.commit()` inside the block if you changed any data.

In [ ]:
# Close the notebook's main cursor and connection
cursor.close()
conn.close()
print("Closed. Connected?", conn.is_connected())

## 14. Practice exercises

Reconnect first (run the next cell), then write each query using `query_df(...)`. Use **placeholders** wherever a value comes from a variable.

1. List the names of all employees who work on the project **ProductX**.
2. For each department, list its name and every location (from `DEPT_LOCATIONS`).
3. Write a function `dependents_of(lname)` that returns the dependents of the employee with that last name, using a `%s` placeholder.
4. Find employees who earn more than **every** employee in department 5.
5. Insert a new project `'Cloudmigration'`, number 40, located in Houston and controlled by department 1. Display it, then delete it. Remember to `commit()`.
6. **Challenge:** inside one transaction, give every employee who works more than 30 hours on any project a $1,000 raise. Print the affected rows, then `rollback()` and confirm the salaries didn't change.

To reset the database at any time, re-run `run_sql_script(conn, SCRIPT_PATH)`.

In [ ]:
conn = mysql.connector.connect(**DB_CONFIG, database="COMPANY")
cursor = conn.cursor()

In [ ]:
# Exercise 1

In [ ]:
# Exercise 2

## Quick reference

```python
conn = mysql.connector.connect(host=..., user=..., password=..., database=...)
cur  = conn.cursor()                    # tuples
cur  = conn.cursor(dictionary=True)     # dicts
cur  = conn.cursor(buffered=True)       # fetch whole result immediately

cur.execute(sql)                        # run one statement
cur.execute(sql, (v1, v2))              # with %s placeholders
cur.execute(sql, {"name": v})           # with %(name)s placeholders
cur.executemany(sql, [(…), (…)])        # same statement, many rows

cur.fetchone()  /  cur.fetchall()  /  cur.fetchmany(n)
cur.column_names   cur.rowcount   cur.lastrowid   cur.statement

conn.commit()   conn.rollback()   conn.autocommit = True
cur.close()     conn.close()
```